# 06. Descarga de NASA POWER (clima diario por municipio)

**Propósito.** Descargar el clima diario de NASA POWER en el centroide de cada municipio del panel: temperatura media y máxima, humedad relativa, viento y precipitación.

**Entradas:** `BDQueimadas/centroides_municipio.csv` (salida del notebook 02). Opcional: `panel_semanal_base.csv` para verificar que no falte ningún municipio.

**Salidas** (`NasaPower/`): `nasa_power_limpio.csv` (final), `nasa_power_parcial.csv` (progreso, se puede borrar al terminar) y `nasa_power_errores.csv`.

**Qué cambió respecto al Hito 2**
- Los puntos de consulta salen de los centroides de los **focos** (antes de DETER); son todos los municipios del panel (alrededor de 542 pares).
- Variables nuevas: `PRECTOTCORR` (precipitación corregida) y `T2M_MAX` (temperatura máxima), además de `T2M`, `RH2M` y `WS2M`.
- Periodo: desde 2021-09-01 (holgura para ventanas de hasta 120 días en el notebook 07) hasta 2026-10-25.
- Robustez para Colab: checkpoint cada 25 municipios con reanudación, 3 reintentos con espera creciente y registro de errores.

Es el notebook más lento (una llamada a la API por municipio). Si Colab se desconecta, se vuelve a ejecutar y continúa donde quedó.

In [1]:
from pathlib import Path
import unicodedata

import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# 1. Entorno: Google Colab (Drive) o ejecucion local
# ---------------------------------------------------------------------------
USAR_DRIVE = True   # True en Colab; False para trabajar con la carpeta local del repositorio
BASE_DRIVE = Path("/content/drive/MyDrive/BIODIVERSITY_GUARD")

if USAR_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        BASE = BASE_DRIVE
    except ImportError:
        print("No se detecto Google Colab: se usa la raiz local del repositorio.")
        USAR_DRIVE = False

if not USAR_DRIVE:
    # la raiz es la primera carpeta (hacia arriba desde la actual) que contiene "datos"
    _aqui = Path.cwd().resolve()
    BASE = next((p for p in [_aqui, *_aqui.parents] if (p / "datos").is_dir()), None)
    if BASE is None:
        raise FileNotFoundError("No se encontro la carpeta 'datos' subiendo desde " + str(_aqui))

if not BASE.exists():
    raise FileNotFoundError(f"No existe BASE = {BASE}. Verifica la carpeta BIODIVERSITY_GUARD en Drive.")

# ---------------------------------------------------------------------------
# 2. Rutas (todas salen de BASE)
# ---------------------------------------------------------------------------
DIR_PREP = BASE / "datos" / "datasets_preprocesamiento"
DIR_BDQ = DIR_PREP / "BDQueimadas"
DIR_DETER = DIR_PREP / "DETER"
DIR_NASA = DIR_PREP / "NasaPower"
DIR_SISAM = DIR_PREP / "SISAM"
DIR_EDA = BASE / "datos" / "datasets_eda"

# ---------------------------------------------------------------------------
# 3. Constantes comunes (identicas en los 7 notebooks)
# ---------------------------------------------------------------------------
FECHA_INICIO_DATOS = "2022-01-01"
FECHA_FIN_EXTRACCION = "2026-11-01"      # DETER y BDQueimadas
PRIMERA_SEMANA = "2022-01-03"             # primer lunes con semana completa (2022-01-01 es sabado)
ULTIMA_SEMANA = "2026-10-19"              # lunes de la ultima semana completa (termina domingo 2026-10-25)
SATELITE_REFERENCIA = "AQUA_M-T"
ESTADOS_AMAZONIA_LEGAL = ["ACRE", "AMAPÁ", "AMAZONAS", "MARANHÃO", "MATO GROSSO", "PARÁ", "RONDÔNIA", "RORAIMA", "TOCANTINS"]
MAPA_UF = {"ACRE": "AC", "AMAPÁ": "AP", "AMAZONAS": "AM", "MARANHÃO": "MA", "MATO GROSSO": "MT", "PARÁ": "PA", "RONDÔNIA": "RO", "RORAIMA": "RR", "TOCANTINS": "TO"}
CORRECCIONES_MUNICIPIO = {"ELDORADO DOS CARAJAS": "ELDORADO DO CARAJAS", "SANTA ISABEL DO PARA": "SANTA IZABEL DO PARA"}
FECHA_CORTE_CENTROIDES = "2024-12-31"     # los centroides se fijan con focos hasta esta fecha

# ---------------------------------------------------------------------------
# 4. Funciones comunes
# ---------------------------------------------------------------------------
def normalizar_texto(texto):
    # mayusculas, sin tildes ni espacios sobrantes
    if pd.isna(texto):
        return texto
    texto = str(texto).upper().strip()
    return unicodedata.normalize("NFKD", texto).encode("ASCII", "ignore").decode("utf-8")


def semana_lunes(fecha):
    # lunes de la semana (lunes a domingo) de cada fecha; recibe una Serie de fechas
    f = pd.to_datetime(fecha).dt.normalize()
    return f - pd.to_timedelta(f.dt.day_of_week, unit="D")


SEMANAS = pd.date_range(PRIMERA_SEMANA, ULTIMA_SEMANA, freq="W-MON")   # calendario de semanas completas
assert len(SEMANAS) == 251, f"El calendario deberia tener 251 semanas y tiene {len(SEMANAS)}"
assert (SEMANAS.day_of_week == 0).all(), "Todas las semanas deben empezar en lunes"


def verificar_entradas(archivos):
    # imprime los archivos de entrada con su tamano y falla si falta alguno
    faltan = []
    for a in map(Path, archivos):
        if a.exists():
            print(f"  OK  {a.relative_to(BASE)}  ({a.stat().st_size / 1e6:,.1f} MB)")
        else:
            faltan.append(a)
    if faltan:
        lista = "\n".join(f"  - {a}" for a in faltan)
        raise FileNotFoundError("Faltan archivos de entrada (copialos a esas carpetas y vuelve a ejecutar):\n" + lista)


print("BASE:", BASE)
print("Calendario:", SEMANAS[0].date(), "a", SEMANAS[-1].date(), f"({len(SEMANAS)} semanas)")

import time
import requests

RUTA_CENTROIDES = DIR_BDQ / "centroides_municipio.csv"
RUTA_PANEL = DIR_PREP / "panel_semanal_base.csv"
RUTA_PARCIAL = DIR_NASA / "nasa_power_parcial.csv"
RUTA_ERRORES = DIR_NASA / "nasa_power_errores.csv"
RUTA_SALIDA = DIR_NASA / "nasa_power_limpio.csv"
DIR_NASA.mkdir(parents=True, exist_ok=True)

URL_NASA = "https://power.larc.nasa.gov/api/temporal/daily/point"
VARIABLES_NASA = ["T2M", "RH2M", "WS2M", "PRECTOTCORR", "T2M_MAX"]
NASA_INICIO = "2021-09-01"      # holgura para ventanas de hasta 120 dias (notebook 07)
NASA_FIN = "2026-10-25"         # domingo de la ultima semana completa
N_DIAS = (pd.Timestamp(NASA_FIN) - pd.Timestamp(NASA_INICIO)).days + 1
LOTE_CHECKPOINT = 25            # municipios entre checkpoints
REINTENTOS = 3
ESPERA_BASE_S = 5               # espera creciente: 5, 10, 15 segundos
TIMEOUT_S = 60
PAUSA_S = 0.3                   # pausa entre llamadas para no saturar la API
CLAVE = ["municipio_norm", "uf"]

print("\nArchivos de entrada:")
verificar_entradas([RUTA_CENTROIDES])
print(f"Periodo: {NASA_INICIO} a {NASA_FIN} ({N_DIAS} dias)")

Mounted at /content/drive
BASE: /content/drive/MyDrive/BIODIVERSITY_GUARD
Calendario: 2022-01-03 a 2026-10-19 (251 semanas)

Archivos de entrada:
  OK  datos/datasets_preprocesamiento/BDQueimadas/centroides_municipio.csv  (0.0 MB)
Periodo: 2021-09-01 a 2026-10-25 (1881 dias)


In [2]:
centroides = pd.read_csv(RUTA_CENTROIDES)
print("Municipios a consultar:", len(centroides))
assert not centroides.duplicated(CLAVE).any()

if RUTA_PANEL.exists():
    pares_panel = pd.read_csv(RUTA_PANEL, usecols=CLAVE).drop_duplicates()
    faltan = set(map(tuple, pares_panel.values)) - set(map(tuple, centroides[CLAVE].values))
    assert not faltan, f"{len(faltan)} municipios del panel sin centroide. Primeros: {sorted(faltan)[:10]}"
    print(f"OK: los {len(pares_panel)} municipios del panel tienen centroide")
else:
    print("Aun no existe panel_semanal_base.csv (notebook 03): no se pudo verificar la cobertura del panel.")
centroides.head()

Municipios a consultar: 543
OK: los 543 municipios del panel tienen centroide


,municipio_norm,uf,lat,lon,n_focos,centroide_fuente
0,ABAETETUBA,PA,-1.773752,-48.785494,240,2022_2024
1,ABEL FIGUEIREDO,PA,-4.931251,-48.432482,115,2022_2024
2,ACAILANDIA,MA,-4.794587,-47.423763,381,2022_2024
3,ACARA,PA,-2.057389,-48.384678,1856,2022_2024
4,ACRELANDIA,AC,-9.938650,-66.902844,391,2022_2024


## Llamada de prueba y variables disponibles

Se consulta el primer municipio con las cinco variables. Si alguna no viene en la respuesta (o la API rechaza la combinación), se prueba variable por variable y se **continúa con las disponibles**, avisando de forma muy visible cuál falta. El notebook 07 exige las cinco, así que una variable faltante se verá ahí también.

In [3]:
def consultar_nasa(lat, lon, variables):
    # una llamada a la API con reintentos y espera creciente; devuelve el JSON
    params = {
        "parameters": ",".join(variables), "community": "AG",
        "longitude": lon, "latitude": lat,
        "start": pd.Timestamp(NASA_INICIO).strftime("%Y%m%d"), "end": pd.Timestamp(NASA_FIN).strftime("%Y%m%d"),
        "format": "JSON",
    }
    ultimo = None
    for intento in range(1, REINTENTOS + 1):
        try:
            r = requests.get(URL_NASA, params=params, timeout=TIMEOUT_S)
            if r.status_code == 200:
                return r.json()
            raise RuntimeError(f"HTTP {r.status_code}: {r.text[:150]}")
        except Exception as e:
            ultimo = e
            if intento < REINTENTOS:
                time.sleep(ESPERA_BASE_S * intento)
    raise RuntimeError(f"Fallo tras {REINTENTOS} intentos: {ultimo}")


def variables_en_respuesta(data):
    return set(data["properties"]["parameter"].keys())


def descubrir_variables(lat, lon):
    # devuelve la lista de variables que la API entrega para un punto
    try:
        presentes = variables_en_respuesta(consultar_nasa(lat, lon, VARIABLES_NASA))
        return [v for v in VARIABLES_NASA if v in presentes]
    except Exception as e:
        print("La consulta conjunta fallo, se prueba variable por variable:", e)
        ok = []
        for v in VARIABLES_NASA:
            try:
                if v in variables_en_respuesta(consultar_nasa(lat, lon, [v])):
                    ok.append(v)
            except Exception as e2:
                print(f"  {v}: no disponible ({e2})")
        return ok


fila0 = centroides.iloc[0]
VARIABLES_OK = descubrir_variables(fila0["lat"], fila0["lon"])
faltantes = [v for v in VARIABLES_NASA if v not in VARIABLES_OK]
if faltantes:
    print("!" * 70)
    print(f"!!! ATENCION: NASA POWER NO ENTREGO LAS VARIABLES: {faltantes}")
    print("!!! Se continua con:", VARIABLES_OK)
    print("!" * 70)
else:
    print("OK: las cinco variables estan en la respuesta:", VARIABLES_OK)
assert VARIABLES_OK, "No hay ninguna variable disponible"
COLUMNAS_SALIDA = ["fecha", "municipio_norm", "uf"] + VARIABLES_OK

OK: las cinco variables estan en la respuesta: ['T2M', 'RH2M', 'WS2M', 'PRECTOTCORR', 'T2M_MAX']


In [4]:
def json_a_dataframe(data, municipio_norm, uf, variables):
    # convierte la respuesta en una fila por dia; el codigo de relleno -999 pasa a NaN
    parametros = data["properties"]["parameter"]
    d = pd.DataFrame({v: parametros[v] for v in variables})
    d = d.reset_index().rename(columns={"index": "fecha_str"})
    d["fecha"] = pd.to_datetime(d["fecha_str"], format="%Y%m%d")
    d["municipio_norm"], d["uf"] = municipio_norm, uf
    for v in variables:
        d[v] = d[v].replace(-999, np.nan)
    return d[["fecha", "municipio_norm", "uf"] + variables]


# prueba con el primer municipio
data_prueba = consultar_nasa(fila0["lat"], fila0["lon"], VARIABLES_OK)
df_prueba = json_a_dataframe(data_prueba, fila0["municipio_norm"], fila0["uf"], VARIABLES_OK)
print(df_prueba.shape, "| filas esperadas:", N_DIAS)
assert len(df_prueba) == N_DIAS, "La API no devolvio todos los dias pedidos"
print(df_prueba.tail(3))

(1881, 8) | filas esperadas: 1881
          fecha municipio_norm  uf    T2M   RH2M  WS2M  PRECTOTCORR  T2M_MAX
1878 2026-10-23     ABAETETUBA  PA  31.61  51.67  1.03         0.00    37.88
1879 2026-10-24     ABAETETUBA  PA  31.71  53.92  1.20         0.40    37.56
1880 2026-10-25     ABAETETUBA  PA  30.98  57.95  1.22         2.02    36.72


### Hallazgos de esta etapa

**Qué se observa**
- La API devolvió las **5 variables**: `T2M`, `RH2M`, `WS2M`, `PRECTOTCORR` y `T2M_MAX`.
- La consulta de prueba trae **1,881** días (2021-09-01 a 2026-10-25), igual que lo esperado.

**Por qué se decidió así**
- El periodo empieza el **1 de septiembre de 2021**, antes de la primera semana del panel. Así las ventanas móviles de hasta **120 días** (notebook 07) están completas desde la semana 1.
- Las consultas usan el centroide de cada municipio, que sale de los focos.


## Descarga con checkpoint y reanudación

Cada 25 municipios se agregan sus filas a `nasa_power_parcial.csv`. Al reiniciar, un municipio cuenta como descargado solo si tiene todos sus días (`N_DIAS` filas); si el archivo quedó cortado a la mitad, ese municipio se vuelve a descargar. Los errores se registran en una lista y se guardan; al final el notebook se detiene si quedó algún municipio sin descargar.

In [5]:
def cargar_parcial():
    # municipios completos del archivo parcial (los incompletos se eliminan para volver a bajarlos)
    if not RUTA_PARCIAL.exists():
        return set()
    p = pd.read_csv(RUTA_PARCIAL, parse_dates=["fecha"])
    if list(p.columns) != COLUMNAS_SALIDA:
        raise ValueError(f"El parcial tiene columnas {list(p.columns)} y se esperaban {COLUMNAS_SALIDA}: borra {RUTA_PARCIAL.name} para empezar de cero")
    conteo = p.groupby(CLAVE).size()
    completos = set(conteo[conteo == N_DIAS].index)
    if len(completos) != len(conteo):
        idx = pd.MultiIndex.from_frame(p[CLAVE])
        p = p[idx.isin(list(completos))]
        p.to_csv(RUTA_PARCIAL, index=False, encoding="utf-8")
        print(f"Se descartaron {len(conteo) - len(completos)} municipios incompletos del parcial")
    return completos


def guardar_lote(lote):
    if not lote:
        return
    nuevo = pd.concat(lote, ignore_index=True)
    nuevo.to_csv(RUTA_PARCIAL, mode="a", header=not RUTA_PARCIAL.exists(), index=False, encoding="utf-8")


completos = cargar_parcial()
pendientes = centroides[[tuple(x) not in completos for x in centroides[CLAVE].values]].reset_index(drop=True)
print(f"Ya descargados: {len(completos)} | pendientes: {len(pendientes)}")

errores, lote = [], []
for i, fila in enumerate(pendientes.itertuples(index=False), start=1):
    try:
        data = consultar_nasa(fila.lat, fila.lon, VARIABLES_OK)
        d = json_a_dataframe(data, fila.municipio_norm, fila.uf, VARIABLES_OK)
        if len(d) != N_DIAS:
            raise RuntimeError(f"Se esperaban {N_DIAS} dias y llegaron {len(d)}")
        lote.append(d)
    except Exception as e:
        errores.append({"municipio_norm": fila.municipio_norm, "uf": fila.uf, "error": str(e)[:300]})
    if i % LOTE_CHECKPOINT == 0 or i == len(pendientes):
        guardar_lote(lote)
        lote = []
        print(f"Progreso: {i}/{len(pendientes)} procesados | errores acumulados: {len(errores)}")
    time.sleep(PAUSA_S)

pd.DataFrame(errores, columns=["municipio_norm", "uf", "error"]).to_csv(RUTA_ERRORES, index=False, encoding="utf-8")
print(f"\nCompletado. Errores: {len(errores)} (guardados en {RUTA_ERRORES.name})")

Ya descargados: 0 | pendientes: 543
Progreso: 25/543 procesados | errores acumulados: 0
Progreso: 50/543 procesados | errores acumulados: 0
Progreso: 75/543 procesados | errores acumulados: 0
Progreso: 100/543 procesados | errores acumulados: 0
Progreso: 125/543 procesados | errores acumulados: 0
Progreso: 150/543 procesados | errores acumulados: 0
Progreso: 175/543 procesados | errores acumulados: 0
Progreso: 200/543 procesados | errores acumulados: 0
Progreso: 225/543 procesados | errores acumulados: 0
Progreso: 250/543 procesados | errores acumulados: 0
Progreso: 275/543 procesados | errores acumulados: 0
Progreso: 300/543 procesados | errores acumulados: 0
Progreso: 325/543 procesados | errores acumulados: 0
Progreso: 350/543 procesados | errores acumulados: 0
Progreso: 375/543 procesados | errores acumulados: 0
Progreso: 400/543 procesados | errores acumulados: 0
Progreso: 425/543 procesados | errores acumulados: 0
Progreso: 450/543 procesados | errores acumulados: 0
Progreso: 475

In [6]:
completos = cargar_parcial()
sin_descargar = [tuple(x) for x in centroides[CLAVE].values if tuple(x) not in completos]
assert not sin_descargar, (f"Quedaron {len(sin_descargar)} municipios sin descargar (ver {RUTA_ERRORES.name}). "
                           f"Vuelve a ejecutar esta celda y la anterior. Primeros: {sin_descargar[:10]}")

df_nasa = pd.read_csv(RUTA_PARCIAL, parse_dates=["fecha"])[COLUMNAS_SALIDA]
assert not df_nasa.duplicated(["fecha"] + CLAVE).any(), "Hay filas duplicadas (fecha, municipio)"
assert len(df_nasa) == len(centroides) * N_DIAS

print("Shape:", df_nasa.shape, "| municipios:", df_nasa[CLAVE].drop_duplicates().shape[0])
print("Rango:", df_nasa["fecha"].min().date(), "a", df_nasa["fecha"].max().date())
print("\nNulos por columna (el -999 se convirtio a NaN):")
print(df_nasa[VARIABLES_OK].isna().sum())

ultimos = df_nasa[df_nasa["fecha"] > pd.Timestamp(NASA_FIN) - pd.Timedelta(days=30)]
print("\nPorcentaje de NaN en los ultimos 30 dias:")
print((ultimos[VARIABLES_OK].isna().mean() * 100).round(2))
print("\nEstadisticas:")
print(df_nasa[VARIABLES_OK].describe())

Shape: (1021383, 8) | municipios: 543
Rango: 2021-09-01 a 2026-10-25

Nulos por columna (el -999 se convirtio a NaN):
T2M            0
RH2M           0
WS2M           0
PRECTOTCORR    0
T2M_MAX        0
dtype: int64

Porcentaje de NaN en los ultimos 30 dias:
T2M            0.0
RH2M           0.0
WS2M           0.0
PRECTOTCORR    0.0
T2M_MAX        0.0
dtype: float64

Estadisticas:
                T2M          RH2M          WS2M   PRECTOTCORR       T2M_MAX
count  1.021383e+06  1.021383e+06  1.021383e+06  1.021383e+06  1.021383e+06
mean   2.745746e+01  7.528932e+01  6.211306e-01  4.346333e+00  3.217065e+01
std    2.348243e+00  1.701426e+01  7.761985e-01  7.134941e+00  3.810987e+00
min    1.052000e+01  1.546000e+01  0.000000e+00  0.000000e+00  1.244000e+01
25%    2.586000e+01  6.417000e+01  7.000000e-02  7.000000e-02  2.916000e+01
50%    2.695000e+01  8.138000e+01  2.800000e-01  1.410000e+00  3.113000e+01
75%    2.889000e+01  8.899000e+01  9.600000e-01  5.630000e+00  3.520000e+01
max    3

### Hallazgos de esta etapa

**Qué se observa**
- Se descargaron los **543** municipios con **0 errores**. El resultado tiene **1,021,383** filas (543 municipios x 1,881 días).
- Hay **0 nulos** en las cinco variables y **0 %** de NaN en los últimos 30 días.
- Los valores son plausibles: `T2M` promedia **27.46** °C, `RH2M` **75.29** %, y `PRECTOTCORR` **4.35** mm por día, con un máximo de **173.73** mm.

**Por qué se decidió así**
- La descarga guarda un checkpoint cada 25 municipios y se puede reanudar si Colab se desconecta.
- El código de relleno `-999` se convierte a nulo, aunque en esta corrida no apareció ninguno.

**Qué implica para el modelado**
- El clima de NASA POWER no necesita imputación.


In [7]:
df_nasa.to_csv(RUTA_SALIDA, index=False, encoding="utf-8", date_format="%Y-%m-%d")
print("Guardado:", RUTA_SALIDA.relative_to(BASE), df_nasa.shape)

Guardado: datos/datasets_preprocesamiento/NasaPower/nasa_power_limpio.csv (1021383, 8)
